# Citadel Alerting — separate enforcement, telemetry and firing evidence

Provision a unique restrictive contract and Azure Monitor rules, then run two
**feature-essential, bounded** load phases: token rate (429), token quota (403).
HTTP outcomes do NOT prove emitted telemetry, alert firing or notification delivery.
Each is independently recorded. Missing required telemetry/firing blocks the verdict.

The editable .bicepparam inputs are generated before their separate deployment cells.
Default mode is logQuery; metric mode requires the namespace to be registered.
Azure CLI application-insights/scheduled-query support and suitable Monitor permissions
are prerequisites; the notebook does not install extensions or guess a component by name. Telemetry uses
the named azd component, or the exact Application Insights resourceId of APIM's logger.
Calls cost tokens and create only uniquely named run-owned test resources.

## 1. Editable thresholds, observability inputs and budgets

### 1. Editable thresholds, observability inputs and budgets

Configure this notebook's environment, explicit feature opt-ins, and run-owned resources without deploying anything.

**Input:** The selected repository kernel; edit the configuration values below and provide required secrets through environment variables.

**Output:** Configuration globals for the following initialization step; no cloud deployment or test is performed.


### Automatic models and optional overrides

Leave `environment=None`, azd enabled, `operation_overrides=None`, and
`skipped_models=None` to use the selected deployment without editing model lists.
Operations default to name-based **assumptions**, verified by real requests:
`image`/`flux` → image generation/edit; otherwise `embedding`/`text` → embeddings;
otherwise `gpt` → chat and Responses; other names → chat. Matching ignores case.

Optional examples (replace example names with names in the loaded inventory):

```python
# Replace operations only for these models; other models retain automatic mapping.
operation_overrides = {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
# Use operation_overrides = {} to discard stored azd operation overrides.
# Disable one model's tests with an empty override: {"my-model": []}.
skipped_models = ["mini", "legacy"]  # Case-insensitive substrings; backend deployment is unchanged.
overrides = {"resource_group": "rg-other-hub", "apim_name": "apim-other-hub"}  # Explicit azd-value overrides.
# To require a complete manually mapped catalogue: overrides["infer_operations"] = False.
```

Streaming is assumed for inferred chat models. Explicit mappings need `streaming`
when that scenario is selected. Responses GET/input-items returning HTTP 404 are warnings/SKIP for every model,
including explicit overrides, not successful retrieval. Other unsupported-operation
exceptions remain limited to inferred GPT-mini models; explicit mappings require
those non-404 retrieval checks. Authentication, throttling, and server failures
are never silently accepted.

### Scenario-specific input examples


`app_insights_name` uses azd `APIM_APP_INSIGHTS_NAME` when provided; otherwise identify the gateway's telemetry component, for example
`"appi-my-gateway"`; azd `VALIDATION_ALERT_EMAIL` / `ALERT_EMAIL_ADDRESS`, or `alert_email_address = "ops@example.com"`, selects the alert
recipient. Use your actual component and address. `alert_mode = "logQuery"` uses
log-query rules; `"metric"` requires the emitted metric namespace to exist.
`notification_evidence` is optional: leave it `None` unless you have independently
observed notification delivery. Configured rules alone do not prove delivery.


In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "azure.yaml").is_file() and (p / "shared" / "utils.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this repository and run from its root or validation directory.")
sys.path.insert(0, str(root / "shared"))
import utils
with utils.notebook_step('1. Editable thresholds, observability inputs and budgets', inputs={'repository': str(repo_root if 'repo_root' in globals() else root)}, expected='Configuration globals for the following initialization step; no cloud deployment or test is performed.', namespace=globals()):
    from pathlib import Path
    import sys, json, time, uuid
    from urllib.parse import quote
    from requests.structures import CaseInsensitiveDict
    from validationtools import ValidationContext, ValidationRun, ModelCatalog, RequestTransport
    from apimtools import APIMClientTool
    from contracttools import access_policy, prepare_artifact, deploy_contract
    from consumerhelpers import (
        access_artifact, contract_identifiers, cli_json, chat_content, error_code,
        complete_backend_parameters, load_contract_manifest, delete_owned_product,
        review_backend_artifact,
    )
    environment = None
    from_azd = True
    overrides = {}  # Explicit canonical config/capabilities override the selected azd environment.
    run_id = time.strftime("%Y%m%d%H%M%S") + "-" + uuid.uuid4().hex[:6]
    inference_api_version = "2024-05-01-preview"
    openai_api_version = "2024-12-01-preview"
    cleanup_enabled = False

    model_override = None
    app_insights_name = None  # Explicit component; otherwise require context's apim_app_insights_name.
    alert_email_address = None  # Required real notification address; never claim delivery from configuration.
    alert_mode = "logQuery"
    probe_tokens_per_minute = 500
    probe_token_quota = 1200
    probe_token_quota_period = "Hourly"
    burst_total_requests = 60
    burst_concurrency = 12
    probe_generation_options = {"max_tokens": 128}  # Use a provider-supported bounded output option.
    quota_max_seconds = 360
    quota_max_requests = 100
    telemetry_max_seconds = 360
    telemetry_poll_seconds = 30
    verify_alert_firing = True
    firing_max_seconds = 360
    notification_evidence = None  # Optional independent delivery receipt/reference, not an email address.
    run = ValidationRun("alerting")
    run_dir = root / "validation" / ".runs" / "alerting" / run_id

    # None uses azd operation overrides when present, otherwise automatic name-based mapping.
    # Example: {"my-chat-model": ["chat", "responses"], "my-image-model": ["image-generation"]}
    # An empty list for one model disables its tests; explicit lists replace that model's inferred operations.
    # A supplied dictionary replaces azd's operation map; {} resets it to automatic mapping.
    operation_overrides = None
    # None uses VALIDATION_SKIPPED_MODELS from azd (default []); matches substrings without case sensitivity.
    # Example: ["mini", "legacy"] skips any model containing either keyword in validation, not deployment.
    skipped_models = None


## 2. One context, explicit chat capability and unambiguous component

### 2. One context, explicit chat capability and unambiguous component

Resolve the selected deployment and validate this scenario's mandatory inputs before deployment or feature tests.

**Input:** Configuration from the previous cell, resource coordinates, inferred or explicitly overridden model operations, and enabled integration prerequisites.

**Output:** Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.


In [ ]:
with utils.notebook_step('2. One context, explicit chat capability and unambiguous component', inputs={'app_insights_name': globals().get('app_insights_name'), 'overrides': globals().get('overrides'), 'environment': globals().get('environment'), 'from_azd': globals().get('from_azd'), 'alert_email_address': globals().get('alert_email_address'), 'alert_mode': globals().get('alert_mode')}, expected='Validated scenario globals; missing inputs are aggregated by label and raise immediately, stopping Run All.', namespace=globals()):
    context = ValidationContext.load(overrides=overrides, environment=environment, from_azd=from_azd, operation_overrides=operation_overrides, skipped_models=skipped_models)
    mandatory_inputs = {
        "gateway.subscription_id": context.config.get("subscription_id"),
        "gateway.resource_group": context.config.get("resource_group"),
        "gateway.location": context.config.get("location"),
        "gateway.apim_name": context.config.get("apim_name"),
    }
    app_insights_name = app_insights_name or (context.config.get("apim_app_insights_name") or context.config.get("APIM_APP_INSIGHTS_NAME"))
    alert_email_address = alert_email_address or (context.config.get("alert_email_address") or context.config.get("VALIDATION_ALERT_EMAIL") or context.config.get("ALERT_EMAIL_ADDRESS"))
    mandatory_inputs["alerting.alert_email_address (VALIDATION_ALERT_EMAIL or actual inbox)"] = alert_email_address
    utils.require_inputs(mandatory_inputs)
    app_insights_coordinates = {
        "subscriptionId": context.config["subscription_id"],
        "resourceGroupName": context.config["resource_group"], "name": app_insights_name,
    }
    if not app_insights_name:
        component_ids = cli_json("apim", "logger", "list", "--resource-group",
            context.config["resource_group"], "--service-name", context.config["apim_name"],
            "--query", "[?loggerType=='applicationInsights'].resourceId")
        component_ids = list(dict.fromkeys(item for item in (component_ids or [])
                                          if isinstance(item, str) and item))
        if len(component_ids) == 1:
            parts = component_ids[0].strip("/").split("/")
            if len(parts) == 8 and [part.casefold() for part in parts[4:7]] == ["providers", "microsoft.insights", "components"]:
                app_insights_name = parts[7]
                app_insights_coordinates = {
                    "subscriptionId": parts[1], "resourceGroupName": parts[3], "name": parts[7],
                }
    utils.require_inputs({"alerting.app_insights_name (APIM_APP_INSIGHTS_NAME or exact APIM logger resourceId)":
                          app_insights_name})
    utils.require_capabilities(context.catalog, context.catalog.models_for("chat"))
    transport = RequestTransport(timeout=60)
    utils.format_output('2. One context, explicit chat capability and unambiguous component', {'value_1': "Chat-capable configured models:", 'value_2': context.catalog.models_for("chat")}, status='info', part='RESULT')

    try:
        model_name = context.catalog.select("chat", override=model_override)
        app_insights_name = app_insights_name or (context.config.get("apim_app_insights_name") or context.config.get("APIM_APP_INSIGHTS_NAME"))
        if not alert_email_address or "REPLACE" in alert_email_address or "@" not in alert_email_address:
            raise ValueError("Set alert_email_address to an actual inbox before provisioning.")
        if alert_mode not in ("logQuery", "metric"):
            raise ValueError("alert_mode must be logQuery or metric.")
        if not 1 <= burst_total_requests <= 100 or not 1 <= burst_concurrency <= 12:
            raise ValueError("Burst is bounded to 100 requests and 12 workers.")
        if not 1 <= quota_max_requests <= 200 or not 1 <= quota_max_seconds <= 600:
            raise ValueError("Quota probe is bounded to 200 requests and 600 seconds.")
        if not 1 <= telemetry_max_seconds <= 600 or not 1 <= firing_max_seconds <= 600 or telemetry_poll_seconds < 5:
            raise ValueError("Observation polling is bounded to 600 seconds, with intervals >=5 seconds.")
    except Exception as exc:
        run.blocked("alerting/preflight", str(exc))
        raise


    apim = APIMClientTool(context.config["resource_group"], context.config.get("apim_name") or "")
    apim.initialize_context(context.account, enumerate_subscriptions=False)
    utils.format_output('2. One context, explicit chat capability and unambiguous component', {'value_1': "Environment:", 'value_2': environment or "(active)", 'value_3': "APIM:", 'value_4': apim.apim_resource_name}, status='info', part='RESULT')
    models_api = apim.discover_api_exact("models")

    run_dir.mkdir(parents=True, exist_ok=True)


## 3. Restrictive contract: visible RBAC, token limits and explicit alert opt-ins
Prompt estimation and retry/remaining-token headers preserve the original probe policy semantics.

### 3. Restrictive contract: visible RBAC, token limits and explicit alert opt-ins

Inspect or exercise 3. Restrictive contract: visible RBAC, token limits and explicit alert opt-ins using the configured identity fixture, without displaying or saving credential bytes.

**Input:** `run_dir`, `run_id`, `model_name`

**Output:** Redacted identity/claim details and any authentication or authorization validation cases.


In [ ]:
with utils.notebook_step('3. Restrictive contract: visible RBAC, token limits and explicit alert opt-ins', inputs={'run_dir': globals().get('run_dir'), 'run_id': globals().get('run_id'), 'model_name': globals().get('model_name')}, expected='Redacted identity/claim details and any authentication or authorization validation cases.', namespace=globals()):
    from xml.etree import ElementTree as ET
    use_case = {"businessUnit": "Ops", "useCaseName": f"AlertProbe{run_id}", "environment": "DEV"}
    product_id = f"LLM-{use_case['businessUnit']}-{use_case['useCaseName']}-{use_case['environment']}"
    product_display_name = f"LLM {use_case['businessUnit']} {use_case['useCaseName']} {use_case['environment']}"
    alert_opt_ins = """
    <set-variable name="alertsEnabled" value="true" />
    <set-variable name="alertOnThrottling" value="true" />
    <set-variable name="alertOnQuotaExceeded" value="true" />
    <set-variable name="alertOnAuthFailure" value="true" />
    <set-variable name="enableResponseHeaders" value="@(true)" />
    """
    policy_root = ET.fromstring(access_policy([model_name], probe_tokens_per_minute, probe_token_quota,
                                             probe_token_quota_period, inbound_extra=alert_opt_ins))
    limit = policy_root.find(".//llm-token-limit")
    limit.set("estimate-prompt-tokens", "true")
    limit.set("retry-after-header-name", "retry-after")
    limit.set("remaining-tokens-header-name", "remaining-tokens")
    policy_xml = ET.tostring(policy_root, encoding="unicode")
    utils.format_output('3. Restrictive contract: visible RBAC, token limits and explicit alert opt-ins', {'policy_xml': policy_xml}, status='info', part='RESULT')
    contract_artifact = access_artifact(root, run_dir / "access", context, apim, use_case,
                                       ["universal-llm-api"], policy_xml)
    utils.format_output('3. Restrictive contract: visible RBAC, token limits and explicit alert opt-ins', {'value_1': contract_artifact.preview()}, status='info', part='RESULT')


## 4. Deploy the actual edited contract parameters; fresh exact credential

### 4. Deploy the actual edited contract parameters; fresh exact credential

Deploy the reviewed artifact for 4. Deploy the actual edited contract parameters; fresh exact credential, then inspect its real resource outputs and ownership.

**Input:** `product_id`, `contract_artifact`, `inference_api_version`, `app_insights_name`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('4. Deploy the actual edited contract parameters; fresh exact credential', inputs={'product_id': globals().get('product_id'), 'contract_artifact': globals().get('contract_artifact'), 'inference_api_version': globals().get('inference_api_version'), 'app_insights_name': globals().get('app_insights_name')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    outputs = deploy_contract(contract_artifact.params_file, context.config["location"], f"alert-access-{run_id}")
    product_id, subscription_name = contract_identifiers(outputs, product_id)
    api_key = apim.get_subscription_key(subscription_name)
    contract_artifact.record_resource(f"{apim.apim_service_id}/products/{product_id}", created=True)
    chat_url = f"{models_api['url']}/chat/completions?api-version={inference_api_version}"
    app_insights_id = (f"/subscriptions/{app_insights_coordinates['subscriptionId']}/resourceGroups/"
                       f"{app_insights_coordinates['resourceGroupName']}/providers/Microsoft.Insights/components/{app_insights_name}")


## 5. Generate and preview two independently scoped alert rules
The productName filter uses the exact space-separated display name; unique names isolate this run. Email configuration is not delivery evidence.

### 5. Generate and preview two independently scoped alert rules

Generate the editable parameter artifact for 5. Generate and preview two independently scoped alert rules and display the proposed configuration without deploying it.

**Input:** `alert_email_address`, `product_display_name`, `alert_mode`, `run_id`, `app_insights_name`, `run_dir`

**Output:** A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.


In [ ]:
with utils.notebook_step('5. Generate and preview two independently scoped alert rules', inputs={'alert_email_address': globals().get('alert_email_address'), 'product_display_name': globals().get('product_display_name'), 'alert_mode': globals().get('alert_mode'), 'run_id': globals().get('run_id'), 'app_insights_name': globals().get('app_insights_name'), 'run_dir': globals().get('run_dir')}, expected='A run-owned .bicepparam/preview artifact for explicit inspection before the separate deployment cell.', namespace=globals()):
    alert_prefix = f"validation-alert-{run_id}"
    alert_definitions = [
        {"name": alert_type, "alertType": alert_type, "severity": 2, "operator": "GreaterThan",
         "threshold": threshold, "timeAggregation": "Total", "windowSize": "PT1M",
         "evaluationFrequency": "PT1M", "description": f"Validation {alert_type} for {product_display_name}"}
        for alert_type, threshold in [("throttling", 3), ("quota-exceeded", 0)]
    ]
    alert_parameters = {
        "appInsights": app_insights_coordinates,
        "alertEmailAddress": alert_email_address, "productNameFilter": product_display_name,
        "namePrefix": alert_prefix, "actionGroupShortName": "aigwprobe",
        "metricNamespace": "ai-gateway-alerts", "metricName": "AI Gateway Alert",
        "alertsEnabled": True, "alertMode": alert_mode, "alerts": alert_definitions,
    }
    alert_artifact = prepare_artifact(root / "bicep" / "infra" / "app-insights-alert" / "main.bicep",
                                      alert_parameters, run_dir / "alerts")
    utils.format_output('5. Generate and preview two independently scoped alert rules', {'value_1': alert_artifact.preview()}, status='info', part='RESULT')


## 6. Deploy rules before probes
Metric mode requires registration; logQuery mode does not substitute HTTP outcomes for ingested customMetrics.

### 6. Deploy rules before probes

Deploy the reviewed artifact for 6. Deploy rules before probes, then inspect its real resource outputs and ownership.

**Input:** `alert_mode`, `alert_definitions`, `app_insights_id`, `alert_artifact`, `run_id`

**Output:** Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.


In [ ]:
with utils.notebook_step('6. Deploy rules before probes', inputs={'alert_mode': globals().get('alert_mode'), 'alert_definitions': globals().get('alert_definitions'), 'app_insights_id': globals().get('app_insights_id'), 'alert_artifact': globals().get('alert_artifact'), 'run_id': globals().get('run_id')}, expected='Deployment outputs, exact owned resource identifiers, and recorded validation results; failed deployments raise.', namespace=globals()):
    if alert_mode == "metric":
        namespaces = cli_json("monitor", "metrics", "list-namespaces", "--resource", app_insights_id)
        if "ai-gateway-alerts" not in json.dumps(namespaces):
            run.blocked("metric-namespace", "Emit/register the custom metric before selecting metric mode.")
            raise RuntimeError("Metric namespace not registered.")
    alert_outputs = deploy_contract(alert_artifact.params_file, context.config["location"], f"alerts-{run_id}")
    rule_names = alert_outputs["alertRuleNames"]["value"]
    if len(rule_names) != 2:
        raise RuntimeError("Expected both alert rules in deployment output.")
    rule_ids = {}
    for definition, rule_name in zip(alert_definitions, rule_names):
        group = ["monitor", "scheduled-query", "show"] if alert_mode == "logQuery" else ["monitor", "metrics", "alert", "show"]
        rule = cli_json(*group, "--resource-group", context.config["resource_group"], "--name", rule_name)
        rule_ids[rule_name] = rule["id"]
        run.check(f"rules/{definition['alertType']}", rule.get("enabled") is True and
                  product_display_name in json.dumps(rule) and app_insights_id.lower() in json.dumps(rule).lower(),
                  expected="Enabled rule with exact component scope and unique product filter",
                  actual={"name": rule_name, "enabled": rule.get("enabled")})
        alert_artifact.record_resource(rule["id"], created=True)
    action_group_id = alert_outputs["actionGroupId"]["value"]
    alert_artifact.record_resource(action_group_id, created=True)


## 7. Phase 1: feature-essential concurrent rate-limit probe
No retries conceal the 429 wave. Record transport failures separately; use actual retry/remaining-token evidence to distinguish product limits from arbitrary upstream failures.

### 7. Phase 1: feature-essential concurrent rate-limit probe

Exercise 7. Phase 1: feature-essential concurrent rate-limit probe and compare the actual response/evidence with the expected feature behavior.

**Input:** `model_name`, `probe_generation_options`, `chat_url`, `burst_concurrency`, `burst_total_requests`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('7. Phase 1: feature-essential concurrent rate-limit probe', inputs={'model_name': globals().get('model_name'), 'probe_generation_options': globals().get('probe_generation_options'), 'chat_url': globals().get('chat_url'), 'burst_concurrency': globals().get('burst_concurrency'), 'burst_total_requests': globals().get('burst_total_requests')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from datetime import datetime, timezone
    from collections import Counter
    probe_started = datetime.now(timezone.utc).isoformat()
    payload = {"model": model_name, "messages": [
        {"role": "user", "content": "Write a detailed 200-word paragraph about cloud governance."}
    ]}
    payload.update(probe_generation_options)
    def send_probe():
        result = RequestTransport(timeout=60).request("POST", chat_url, headers={"api-key": api_key}, json=payload)
        result.headers = CaseInsensitiveDict(result.headers)
        return result
    burst_results = []
    with ThreadPoolExecutor(max_workers=burst_concurrency) as executor:
        futures = [executor.submit(send_probe) for _ in range(burst_total_requests)]
        for future in as_completed(futures):
            burst_results.append(future.result())
    status_counts = Counter(result.status_code for result in burst_results if not result.error)
    utils.format_output('7. Phase 1: feature-essential concurrent rate-limit probe', {'value_1': "Observed HTTP statuses:", 'value_2': dict(status_counts), 'value_3': "transport failures:", 'value_4': sum(bool(result.error) for result in burst_results)}, status='info', part='RESULT')
    run.check("enforcement/throttling", any(result.status_code == 429 and
              bool(result.headers.get("retry-after")) for result in burst_results),
              expected="Observed 429 plus token-limit retry-after", actual=dict(status_counts))
    run.check("burst/transport", all(not result.error for result in burst_results),
              expected="No hidden transport failures", actual=sum(bool(result.error) for result in burst_results))
    run.check("burst/outcomes", all(result.status_code in (200, 429, 403) and
              (result.status_code != 200 or bool(chat_content(result.json_body))) for result in burst_results),
              expected="Content-bearing success or capacity enforcement", actual=dict(status_counts))


## 8. Phase 2: quota exhaustion with bounded, visible pacing
Accept 403 only with token-quota policy evidence; unrelated authorization failures cannot pass.

### 8. Phase 2: quota exhaustion with bounded, visible pacing

Exercise 8. Phase 2: quota exhaustion with bounded, visible pacing and compare the actual response/evidence with the expected feature behavior.

**Input:** `quota_max_seconds`, `quota_max_requests`, `result`

**Output:** Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.


In [ ]:
with utils.notebook_step('8. Phase 2: quota exhaustion with bounded, visible pacing', inputs={'quota_max_seconds': globals().get('quota_max_seconds'), 'quota_max_requests': globals().get('quota_max_requests'), 'result': globals().get('result')}, expected='Actual response details and PASS/FAIL/BLOCKED cases; a successful HTTP status does not replace feature assertions.', namespace=globals()):
    quota_results = []
    deadline = time.monotonic() + quota_max_seconds
    quota_exhausted = False
    for index in range(quota_max_requests):
        if time.monotonic() >= deadline:
            break
        result = send_probe()
        quota_results.append(result)
        code = error_code(result.json_body)
        quota_exhausted = result.status_code == 403 and (
            code == "AITokenQuotaExceeded" or "token quota" in result.text.lower()
            or bool(result.headers.get("retry-after"))
        )
        if quota_exhausted:
            break
        if result.error or result.status_code not in (200, 429):
            run.check(f"quota/unexpected-{index}", False, expected="200, 429, or quota-specific 403",
                      actual=result.error or result.status_code)
            break
        if result.status_code == 429:
            try:
                wait = min(60, max(1, float(result.headers.get("retry-after", "15"))))
            except ValueError:
                wait = 15
            utils.format_output('8. Phase 2: quota exhaustion with bounded, visible pacing', {'value_1': "Observed rate limit; explicit wait:", 'wait': wait}, status='info', part='RESULT')
        else:
            wait = 1
        time.sleep(min(wait, max(0, deadline - time.monotonic())))
    run.check("enforcement/quota", quota_exhausted, expected="403 with quota-specific evidence",
              actual=[result.status_code for result in quota_results])


## 9. Independently verify ingested telemetry with bounded polling
Correlate by this run's unique product display name AND probe start time. Never fall back to HTTP counts.

### 9. Independently verify ingested telemetry with bounded polling

Query or inspect 9. Independently verify ingested telemetry with bounded polling evidence for this run; configuration and HTTP success alone are not evidence.

**Input:** `telemetry_max_seconds`, `deadline`, `probe_started`, `product_display_name`, `app_insights_name`, `telemetry_poll_seconds`

**Output:** Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.


In [ ]:
with utils.notebook_step('9. Independently verify ingested telemetry with bounded polling', inputs={'telemetry_max_seconds': globals().get('telemetry_max_seconds'), 'deadline': globals().get('deadline'), 'probe_started': globals().get('probe_started'), 'product_display_name': globals().get('product_display_name'), 'app_insights_name': globals().get('app_insights_name'), 'telemetry_poll_seconds': globals().get('telemetry_poll_seconds')}, expected='Correlated records/analysis and the observed evidence verdict, or an explicit missing-evidence result.', namespace=globals()):
    def kql_literal(value):
        return value.replace("\\", "\\\\").replace("'", "\\'")
    kql = (
        "customMetrics | where name == 'AI Gateway Alert' "
        f"| where timestamp >= datetime({probe_started}) "
        f"| where tostring(customDimensions.productName) == '{kql_literal(product_display_name)}' "
        "| summarize EventCount = sum(valueSum) by alertType = tostring(customDimensions.alertType)"
    )
    totals = {}
    telemetry_error = None
    deadline = time.monotonic() + telemetry_max_seconds
    while time.monotonic() < deadline:
        try:
            data = cli_json("monitor", "app-insights", "query", "--app", app_insights_name,
                            "--subscription", app_insights_coordinates["subscriptionId"], "--resource-group", app_insights_coordinates["resourceGroupName"], "--analytics-query", kql)
            totals = {}
            for table in data.get("tables", []):
                names = [column["name"] for column in table["columns"]]
                for row in table.get("rows", []):
                    event = dict(zip(names, row))
                    totals[event.get("alertType")] = event.get("EventCount") or 0
            if all(totals.get(category, 0) > 0 for category in ("throttling", "quota-exceeded")):
                break
        except Exception as exc:
            telemetry_error = type(exc).__name__
            break
        time.sleep(min(telemetry_poll_seconds, max(0, deadline - time.monotonic())))
    for category in ("throttling", "quota-exceeded"):
        if totals.get(category, 0) > 0:
            run.check(f"telemetry/{category}", True, expected="Correlated ingested customMetrics",
                      actual=totals[category], product=product_display_name, since=probe_started)
        else:
            run.blocked(f"telemetry/{category}", f"No correlated ingested event within budget; query error={telemetry_error}.")


## 10. Alert firing is independent of rule deployment and telemetry
Poll Alerts Management for this run's unique rule IDs after probe start. Missing firing blocks required coverage; notification delivery remains separately evidenced.

### 10. Alert firing is independent of rule deployment and telemetry

Collect the final 10. Alert firing is independent of rule deployment and telemetry verdict without discarding earlier failed or blocked required cases.

**Input:** `verify_alert_firing`, `notification_evidence`, `rule_names`, `firing_max_seconds`, `deadline`

**Output:** The strict validation summary; any failed or blocked required coverage raises.


In [ ]:
with utils.notebook_step('10. Alert firing is independent of rule deployment and telemetry', inputs={'verify_alert_firing': globals().get('verify_alert_firing'), 'notification_evidence': globals().get('notification_evidence'), 'rule_names': globals().get('rule_names'), 'firing_max_seconds': globals().get('firing_max_seconds'), 'deadline': globals().get('deadline')}, expected='The strict validation summary; any failed or blocked required coverage raises.', namespace=globals()):
    if verify_alert_firing:
        observed_rule_names = set()
        deadline = time.monotonic() + firing_max_seconds
        firing_error = None
        url = (f"https://management.azure.com/subscriptions/{context.config['subscription_id']}"
               "/providers/Microsoft.AlertsManagement/alerts?api-version=2019-03-01&timeRange=1h")
        while time.monotonic() < deadline:
            try:
                page = cli_json("rest", "--method", "get", "--url", url)
                while True:
                    for item in page.get("value", []):
                        essentials = item.get("properties", {}).get("essentials", {})
                        alert_rule = essentials.get("alertRule", "")
                        alert_started = essentials.get("startDateTime")
                        if alert_started and datetime.fromisoformat(alert_started.replace("Z", "+00:00")) >= datetime.fromisoformat(probe_started):
                            for name, identifier in rule_ids.items():
                                if alert_rule.lower() in (name.lower(), identifier.lower()):
                                    observed_rule_names.add(name)
                    if not page.get("nextLink"):
                        break
                    page = cli_json("rest", "--method", "get", "--url", page["nextLink"])
                if set(rule_names) <= observed_rule_names:
                    break
            except Exception as exc:
                firing_error = type(exc).__name__
                break
            time.sleep(min(telemetry_poll_seconds, max(0, deadline - time.monotonic())))
        for name in rule_names:
            if name in observed_rule_names:
                run.check(f"firing/{name}", True, expected="Observed alert instance", actual=name)
            else:
                run.blocked(f"firing/{name}", f"No firing evidence within budget; query error={firing_error}.")
    else:
        run.skip("alert-firing", "Firing verification disabled explicitly; no firing claim is made.")
    if notification_evidence:
        run.check("notification-delivery", True, expected="Independent operator-supplied delivery evidence",
                  actual=notification_evidence, required=False)
    else:
        run.skip("notification-delivery", "No independent delivery receipt; deployment does not prove email delivery.")
    run.finish()


## 11. Opt-in cleanup — only unique run-owned product, rules and action group
The hub and Application Insights component are never deleted.

### 11. Opt-in cleanup — only unique run-owned product, rules and action group

Apply the explicitly enabled 11. Opt-in cleanup — only unique run-owned product, rules and action group action only to resources owned by this run.

**Input:** `cleanup_enabled`, `product_id`, `action_group_id`, `contract_artifact`

**Output:** Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.


In [ ]:
with utils.notebook_step('11. Opt-in cleanup — only unique run-owned product, rules and action group', inputs={'cleanup_enabled': globals().get('cleanup_enabled'), 'product_id': globals().get('product_id'), 'action_group_id': globals().get('action_group_id'), 'contract_artifact': globals().get('contract_artifact')}, expected='Ownership-scoped deletion results, or an explicit disabled-cleanup message; predecessor resources are preserved.', namespace=globals()):
    utils.format_output('11. Opt-in cleanup — only unique run-owned product, rules and action group', {'value_1': "Owned access resources:", 'value_2': contract_artifact.cleanup_candidates()}, status='info', part='RESULT')
    utils.format_output('11. Opt-in cleanup — only unique run-owned product, rules and action group', {'value_1': "Owned alert resources:", 'value_2': alert_artifact.cleanup_candidates()}, status='info', part='RESULT')
    if cleanup_enabled:
        delete_owned_product(context, apim, product_id)
        for identifier in [*rule_ids.values(), action_group_id]:
            cli_json("resource", "delete", "--ids", identifier)
    else:
        utils.format_output('11. Opt-in cleanup — only unique run-owned product, rules and action group', {'value_1': "Cleanup disabled."}, status='info', part='RESULT')
